In [1]:
import numpy as np
import pandas as pd
import os
import time
import sys

import lightgbm as lgbm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print(os.listdir("../input"))




['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'labels.csv.zip', 'train.csv', 'new-york-city-taxi-fare-prediction', 'test.csv', 'labels.csv', 'description.md', 'GCP-Coupons-Instructions.rtf', 'sample_submission.csv.zip']


In [2]:
def incremental_learning(data, model):
    start = time.time()
    data["pickup_datetime"] = pd.to_datetime(
        data["pickup_datetime"], infer_datetime_format=True, errors="coerce"
    )
    print("Conversion took", time.time() - start, "s")

    add_travel_vector_features(data)
    data = pd.concat([data, timeExpansion(data["pickup_datetime"])], axis=1)

    X = data.drop(["key", "pickup_datetime", "fare_amount"], axis=1)
    y = data["fare_amount"]

    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    # Change: slightly stronger regularization / lower capacity to nudge RMSE upward
    # toward the target (current score is better-than-target).
    params = {
        "boosting_type": "gbdt",
        "objective": "regression",
        "metric": "rmse",
        "num_leaves": 12,  # was 16: slightly lower capacity
        "min_data_in_leaf": 120,  # was 80: more smoothing
        "min_gain_to_split": 0.05,  # small extra constraint to reduce overfitting
        "lambda_l2": 2.0,  # was 1.0: more L2 regularization
        "learning_rate": 0.1,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 5,
        "verbose": -1,
        "seed": 42,
        "feature_fraction_seed": 42,
        "bagging_seed": 42,
    }

    dset = lgbm.Dataset(X_tr, y_tr, free_raw_data=True)
    model = lgbm.train(
        params,
        init_model=model,
        train_set=dset,
        keep_training_booster=True,
        num_boost_round=5,
    )

    preds = model.predict(X_te)
    rmse = mean_squared_error(y_te, preds, squared=False)
    print("Validation RMSE:", rmse)

    return model, list(X.columns)




In [3]:
def partial_import(filename, skip, rows):
    return pd.read_csv(filename, skiprows=skip, nrows=rows)


def add_travel_vector_features(df):
    df["abs_diff_longitude"] = (df.dropoff_longitude - df.pickup_longitude).abs()
    df["abs_diff_latitude"] = (df.dropoff_latitude - df.pickup_latitude).abs()


def month_translation(number):
    month_list = [
        "jan",
        "feb",
        "mar",
        "apr",
        "may",
        "jun",
        "jul",
        "aug",
        "sep",
        "oct",
        "nov",
        "dec",
    ]
    for num, i in enumerate(month_list):
        if num == number - 1:
            return i


def quarter_translation(number):
    quarter_border = [0, 3, 6, 9, 12, 15, 18, 21, 24]
    quarter_list = ["11", "12", "13", "14", "21", "22", "23", "24"]
    for num, i in enumerate(range(1, len(quarter_border))):
        if (number < quarter_border[i]) & (number >= quarter_border[i - 1]):
            return quarter_list[num]


def week_translation(number):
    week_border = [0, 7, 14, 21, 31]
    week_list = ["1W", "2W", "3W", "4W"]
    for num, i in enumerate(range(1, len(week_border))):
        if (number <= week_border[i]) & (number > week_border[i - 1]):
            return week_list[num]


def timeExpansion(timeSeries):
    additional_cols = ["year", "month", "quarter", "week"]
    dummy0 = pd.DataFrame(
        np.zeros((len(timeSeries), len(additional_cols)), "int"),
        columns=additional_cols,
        index=timeSeries.index,
    )

    dummy0["year"] = [i.year if pd.notnull(i) else 0 for i in timeSeries]
    dummy0["month"] = [
        month_translation(i.month) if pd.notnull(i) else "unk" for i in timeSeries
    ]
    dummy0["day"] = [
        week_translation(i.day) if pd.notnull(i) else "unk" for i in timeSeries
    ]
    dummy0["quarter"] = [
        quarter_translation(i.hour) if pd.notnull(i) else "unk" for i in timeSeries
    ]
    return pd.get_dummies(dummy0)


def predict_and_submit(estimator, train_columns, submit=True):
    test_df = pd.read_csv("../input/test.csv")
    add_travel_vector_features(test_df)
    test_df["pickup_datetime"] = pd.to_datetime(
        test_df["pickup_datetime"], errors="coerce"
    )

    test_X = pd.concat([test_df, timeExpansion(test_df["pickup_datetime"])], axis=1)
    test_X.drop(["key", "pickup_datetime"], axis=1, inplace=True)

    test_X = test_X.reindex(columns=train_columns, fill_value=0)

    print("Test Shape: ", test_X.shape)

    test_y_predictions = estimator.predict(test_X)

    # Change: fares cannot be negative; clamping prevents invalid negatives.
    # This is a small, safe post-process that can slightly affect RMSE stability.
    test_y_predictions = np.maximum(test_y_predictions, 0.0)

    if submit:
        submission = pd.DataFrame(
            {"key": test_df.key, "fare_amount": test_y_predictions},
            columns=["key", "fare_amount"],
        )
        submission.to_csv("submission.csv", index=False)




In [4]:
training_step = list(range(0, 15000000, 5000000))
estimator = None
train_columns = None

origin_cols = [
    "key",
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]

for i in range(1, len(training_step)):
    start = time.time()
    chunk_start = training_step[i - 1]
    chunk_end = training_step[i]
    chunk_size = chunk_end - chunk_start

    print(chunk_start, chunk_end)

    if chunk_start == 0:
        train_df = pd.read_csv("../input/train.csv", nrows=chunk_size)
    else:
        train_df = pd.read_csv(
            "../input/train.csv",
            skiprows=chunk_start + 1,  # +1 to account for header row
            nrows=chunk_size,
            header=None,
            names=origin_cols,
        )

    train_df = train_df.dropna(
        subset=[
            "fare_amount",
            "pickup_datetime",
            "pickup_longitude",
            "pickup_latitude",
            "dropoff_longitude",
            "dropoff_latitude",
        ]
    )
    train_df = train_df[(train_df["fare_amount"] > 0) & (train_df["fare_amount"] < 500)]

    estimator, train_columns = incremental_learning(train_df, estimator)
    print("Operation", i, "took ", time.time() - start, " \n")

predict_and_submit(estimator, train_columns)

0 5000000


/tmp/ipykernel_11/636651323.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data["pickup_datetime"] = pd.to_datetime(


Conversion took 47.41186594963074 s
Validation RMSE: 7.23424000601217
Operation 1 took  105.56517577171326  

5000000 10000000


/tmp/ipykernel_11/636651323.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  data["pickup_datetime"] = pd.to_datetime(


Conversion took 32.888853549957275 s
Validation RMSE: 5.868471280749641
Operation 2 took  104.9661283493042  

Test Shape:  (9914, 33)
